## Row data ingestion and Processing
##### under this,we are trying to connect url and download the data and save as csv



In [15]:
import io
from pathlib import Path
import pandas as pd
import requests


RAW_DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2016-01.parquet"

def download_raw_file(raw_dir: Path, url: str = RAW_DATA_URL) -> Path:
    """
    Downloads the remote Parquet file as-is and saves it to raw_dir.
    """
    raw_dir = Path(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)

    filename = url.split("/")[-1]
    raw_file_path = raw_dir / filename

    if raw_file_path.exists():
        print(f"[Ingest] Raw file already exists at {raw_file_path}")
        return raw_file_path

    print(f"[Ingest] Downloading raw file from {url}...")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    response = requests.get(url, stream=True, timeout=120, headers=headers)
    response.raise_for_status()

    # Stream file to disk in 1MB chunks to keep memory usage low
    with open(raw_file_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(f"[Ingest] Raw file saved to {raw_file_path}")
    return raw_file_path

def convert_parquet_to_csv(parquet_path: Path, output_dir: Path, max_rows: int = None) -> Path:
    """
    Reads the downloaded Parquet file from disk and exports it as a CSV.
    """
    parquet_path = Path(parquet_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"{parquet_path.stem}.csv"

    if csv_path.exists():
        print(f"[Convert] CSV already exists at {csv_path}")
        return csv_path

    print(f"[Convert] Loading {parquet_path.name}...")
    df = pd.read_parquet(parquet_path)

    if max_rows:
        df = df.head(max_rows)

    print(f"[Convert] Writing {len(df):,} rows to {csv_path}...")
    df.to_csv(csv_path, index=False)

    print(f"[Convert] Successfully created {csv_path}")
    return csv_path

def fetch_and_convert(raw_dir: Path, processed_dir: Path, url: str = RAW_DATA_URL, max_rows: int = None) -> Path:
    # 1. Download raw parquet to raw_dir
    raw_file = download_raw_file(raw_dir, url=url)

    # 2. Convert raw parquet to CSV in processed_dir
    csv_file = convert_parquet_to_csv(raw_file, processed_dir, max_rows=max_rows)
    return csv_file


"""    old code which directtly tried to download and saved as csv
    def fetch_raw_data_as_csv(output_dir: Path, url: str = RAW_DATA_URL, max_rows: int = None) -> Path:

    Downloads remote Parquet data into memory, converts it to CSV,
    and saves the raw CSV file to output_dir.
  
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Derive CSV filename (e.g., yellow_tripdata_2016-01.csv)
    parquet_filename = url.split("/")[-1]
    csv_filename = Path(parquet_filename).stem + ".csv"
    destination = output_dir / csv_filename

    if destination.exists():
        print(f"[Ingest] CSV file already exists at {destination}")
        return destination

    print(f"[Ingest] Downloading raw Parquet from {url}...")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    response = requests.get(url, stream=True, timeout=120, headers=headers)
    response.raise_for_status()

    # Read binary content directly into pandas from memory
    print("[Ingest] Converting Parquet stream to DataFrame...")
    parquet_buffer = io.BytesIO(response.content)
    df = pd.read_parquet(parquet_buffer)

    if max_rows:
        df = df.head(max_rows)

    # Save to disk as CSV
    print(f"[Ingest] Saving CSV to {destination}...")
    df.to_csv(destination, index=False)

    print(f"[Ingest] Done! Saved {len(df):,} rows to {destination}")
    return destination
    """
if __name__ == "__main__":

    # Base data directory (removed the invalid trailing backslash)
    base_data_dir = Path(r"C:\Users\User\nyc-taxi-data-engineering-project\data")
   
    raw_folder = base_data_dir / "raw"
    processed_folder = base_data_dir / "processed"

    # Run the full pipeline
    # Tip: Set max_rows=100000 for fast local testing, or leave None for the full dataset
    fetch_and_convert(raw_dir=raw_folder, processed_dir=processed_folder, max_rows=None)    

[Ingest] Downloading raw file from https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2016-01.parquet...
[Ingest] Raw file saved to C:\Users\User\nyc-taxi-data-engineering-project\data\raw\yellow_tripdata_2016-01.parquet
[Convert] Loading yellow_tripdata_2016-01.parquet...
[Convert] Writing 10,905,067 rows to C:\Users\User\nyc-taxi-data-engineering-project\data\processed\yellow_tripdata_2016-01.csv...
[Convert] Successfully created C:\Users\User\nyc-taxi-data-engineering-project\data\processed\yellow_tripdata_2016-01.csv
